# GenART Day 2 - RNA-seq: from reads (FASTQ) to a gene count matrix, all in shell commands

Every cell below is **plain shell commands** (no scripts hidden behind them). Each command is printed in green with a `$` in front,
and then its complete output follows, live, in the same cell. Nothing is hidden. (The only exceptions are the **Figure** cells: short Python cells that draw a picture from the tables the shell commands made.)

**How to run:** Connect (top right), then run the cells from top to bottom with the play button (or *Runtime > Run all*).
Setup takes about 2-4 minutes (it installs the tools), the analysis itself about 2-3 minutes. (Tested end to end in 2.5 minutes on 2 CPU cores; a free Colab session can be slower.)
(`$LAB`, `$REPO` and `$THREADS` in the commands are shell variables set in cell 0a: the tool folder, the data repository and the number of CPU cores.)

**The data:** REAL RNA-seq reads of one Holstein-Friesian **calf** (138 days old), two tissues - **liver** and **skeletal muscle** - from the
European Nucleotide Archive (project PRJNA1225222 = GEO GSE289870; runs SRR32383427 and SRR32383411). The original files are 2.3 GB each (23 million read pairs).
Here you work with **100,000 read pairs per tissue (about 4 MB per file)**, and only reads that come from **cattle chromosome 19** (1,771 genes), so every step takes seconds.
Liver and muscle are very different tissues, so the final table should show it: liver genes in one column, muscle genes in the other.

| Part | Steps |
|---|---|
| **0 Setup** | get the data, install the tools |
| **A Look at the files** | 1 FASTQ (reads) - 2 FASTA (reference genome) - 3 GTF (gene annotation) |
| **B Reads to a count matrix** | 4 FastQC - 5 read the FastQC result - 6 MultiQC - 7 fastp trimming - 8 FastQC again - 9 HISAT2 index - 10 HISAT2 alignment - 11 look at the BAM - 12 alignment QC - 13 which strand? - 14 featureCounts - 15 the gene count matrix - 16 look at the matrix |
| **C Figures** | 5 pictures made from the tables above |
| **D Wrap-up** | what we made, questions, where to go next |

The whole pipeline in one picture:

```
 raw FASTQ (R1 + R2)
      |--> FastQC, MultiQC ............ look at the quality
      |--> fastp ...................... remove adapters and bad ends
      |       '--> trimmed FASTQ --> FastQC, MultiQC ... check again
      |--> HISAT2 + samtools .......... align to the genome  --> sorted BAM  --> QC
      |--> featureCounts + GTF ........ count read pairs per gene
              '--> gene_count_matrix.tsv   (rows = genes, columns = samples)   <-- the goal
```

## 0. Setup
**0a. Settings** (this one is Python: it only sets the working folder and makes the shell cells print every command).

In [ ]:
import os

WORK = "/content/RNAseq_calf" if os.path.isdir("/content") else os.path.expanduser("~/RNAseq_calf")   # everything is made in this folder
LAB  = os.path.expanduser("~/genart-rnaseq")                                                         # the tools are installed here
REPO = "https://github.com/PratikPathade/GenartWorkshop_Pratik.git"                                      # the data are in this repository
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

os.environ["PATH"] = LAB + "/env/bin:" + LAB + "/bin:" + os.environ["PATH"]
os.environ["MAMBA_ROOT_PREFIX"] = LAB + "/mamba"
os.environ["LAB"] = LAB
os.environ["REPO"] = REPO
os.environ["THREADS"] = str(len(os.sched_getaffinity(0)))      # number of CPU cores, used by the tools below (-p / -t / --thread)

# every shell cell starts with these lines: stop at the first error, send errors to the same stream, print each command in green
trace = os.path.join(os.path.dirname(WORK), "trace_rnaseq.sh")
open(trace, "w").write(r"""
if [ -z "$TRACE_DONE" ]; then
  export TRACE_DONE=1
  exec 2>&1
  set -e
  trap 'case "$BASH_COMMAND" in "for "*|"[ "*) ;; *) printf "\n\033[1;32m$ %s\033[0m\n" "$BASH_COMMAND";; esac' DEBUG
fi
""")
os.environ["BASH_ENV"] = trace
print("working folder:", os.getcwd())
print("CPU cores     :", os.environ["THREADS"])
print("RAM (GB)      :", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))

**0b. Get the data.** The reads, the reference chromosome and the annotation are in the GitHub repository of this tutorial. We clone it and copy the `data` folder.

In [ ]:
%%bash
rm -rf repo data
git clone --depth 1 $REPO repo
cp -r repo/data data
ls -lh data/reads data/ref

**0c. Install the tools** (2-3 minutes, once per session). We use *micromamba* (a fast installer) to get FastQC, MultiQC, fastp, HISAT2, samtools and featureCounts (part of the *Subread* package)
with the same versions we tested. The long output is the download and install.

In [ ]:
%%bash
mkdir -p $LAB/bin
curl -fsSL -o $LAB/bin/micromamba https://github.com/mamba-org/micromamba-releases/releases/download/2.9.0-0/micromamba-linux-64
chmod +x $LAB/bin/micromamba
micromamba --version

In [ ]:
%%bash
[ -x $LAB/env/bin/hisat2 ] || micromamba create -y -p $LAB/env -c conda-forge -c bioconda python=3.11 fastqc=0.13.0 multiqc=1.35 fastp=1.4.0 hisat2=2.2.3 samtools=1.24 subread=2.1.1

**0d. Check the tools** - a version number for each means they are ready.

In [ ]:
%%bash
fastqc --version
multiqc --version
fastp --version
hisat2 --version | head -n 1
samtools --version | head -n 1
featureCounts -v 2>&1 | grep featureCounts

**0e. Make the folders** for the results. One folder per step keeps the files easy to find.

In [ ]:
%%bash
mkdir -p ref index results/qc_raw results/trim results/qc_trim results/align results/counts
ls

# PART A - Look at the files

Before running any tool, **look at the files**. Three kinds of file go into an RNA-seq pipeline:
the **reads** (FASTQ), the **reference genome** (FASTA) and the **gene annotation** (GTF).

## Step 1 of 16: FASTQ - the raw reads
Every read is 4 lines:
* line 1 `@name` - read name (here: the run number, a counter, the sequencer, lane, tile and x:y position on the flow cell, and `/1` = first read of the pair)
* line 2 - the bases (A C G T; N = the sequencer could not tell)
* line 3 `+` - separator
* line 4 - one quality character per base. **Phred score = ASCII code - 33**; Q30 = 1 error in 1,000 bases

Below are the first **two reads of the liver sample, read 1** (101 bases each). Look at the quality line: most symbols are `F` (the best quality here), a few are `:` or `,` (lower quality) - FastQC and fastp will count them for us.

In [ ]:
%%bash
zcat data/reads/liver_R1.fastq.gz | head -n 8

The library is **paired-end**: every RNA fragment was read from both ends. R2 has the same read names as R1 (ending in `/2` = second read of the pair), in the same order.

In [ ]:
%%bash
zcat data/reads/liver_R2.fastq.gz | head -n 4

How big are the files, and how many reads do they hold (a FASTQ record is 4 lines, so reads = lines / 4)?

In [ ]:
%%bash
ls -lh data/reads
for f in data/reads/*.fastq.gz; do echo "$f   $(zcat $f | wc -l | awk '{print $1/4}') reads"; done

Read length - one line per length, with the number of reads:

In [ ]:
%%bash
zcat data/reads/liver_R1.fastq.gz | awk 'NR%4==2 {print length($0)}' | sort -n | uniq -c

Which quality symbols are used? We split the quality line of the first 10,000 reads into single characters and count them,
then convert each symbol to a Phred score (ASCII code minus 33).

In [ ]:
%%bash
zcat data/reads/liver_R1.fastq.gz | head -n 40000 | awk 'NR%4==0' | fold -w1 | sort | uniq -c | sort -k1,1nr
for c in F : , '#'; do echo "symbol $c  = Q$(( $(printf '%d' "'$c") - 33 ))"; done

A NovaSeq sequencer reports only **four quality values** (Q37, Q25, Q11, Q2), so you will not see a smooth quality scale in this data.

## Step 2 of 16: FASTA - the reference genome
The reference is a text file with one `>name` line and then the sequence. We use **chromosome 19** of the cattle reference genome **ARS-UCD2.0** (Ensembl release 114), 63 million bases.
The whole genome (2.7 billion bases) would need an index that is too large for a free Colab session - chromosome 19 is the same kind of problem, just smaller.

In [ ]:
%%bash
gunzip -c data/ref/chr19.fa.gz > ref/chr19.fa
ls -lh ref
grep '>' ref/chr19.fa

`samtools faidx` makes a small index (`.fai`) so that any piece of the genome can be read quickly: name, length, where it starts in the file, and the line width.
Then we read 90 bases from the middle of chromosome 19.

In [ ]:
%%bash
samtools faidx ref/chr19.fa
cat ref/chr19.fa.fai
samtools faidx ref/chr19.fa 19:20000001-20000090

## Step 3 of 16: GTF - the gene annotation
The GTF file says **where the genes are**: one line per gene, transcript, exon, ... with chromosome, start, end, strand and a list of attributes (gene_id, gene_name, biotype).
featureCounts will use the `exon` lines to decide which gene a read pair belongs to. Here is the annotation of chromosome 19 from Ensembl.

In [ ]:
%%bash
zcat data/ref/chr19.gtf.gz | head -n 5
zcat data/ref/chr19.gtf.gz | grep -v '^#' | head -n 3 | cut -c1-260

What kinds of lines are in the file (column 3), and what kinds of genes (the `gene_biotype` attribute)?

In [ ]:
%%bash
zcat data/ref/chr19.gtf.gz | grep -v '^#' | cut -f3 | sort | uniq -c | sort -k1,1nr
zcat data/ref/chr19.gtf.gz | awk -F'\t' '$3=="gene"' | grep -o 'gene_biotype "[^"]*"' | sort | uniq -c | sort -k1,1nr

One gene in detail: **VTN** (vitronectin, a protein made by the liver). Its gene line and its first exons (columns: chromosome, feature, start, end, strand):

In [ ]:
%%bash
zcat data/ref/chr19.gtf.gz | grep 'gene_name "VTN"' | awk -F'\t' '$3=="gene" || $3=="exon"' | cut -f1,3,4,5,7 | head -n 8

# PART B - From reads to a gene count matrix

## Step 4 of 16: FastQC - quality control of the raw reads
**FastQC** reads a FASTQ file and tests it with a set of modules (per-base quality, base composition, adapters, duplicates, ...). It writes one HTML report and one zip file per FASTQ file.
`-t` is the number of threads, `--outdir` the output folder, `--extract` also unpacks the zip file so that we can read the text results. The "Approx ... % complete" lines are its progress.

In [ ]:
%%bash
fastqc -t $THREADS --extract --outdir results/qc_raw data/reads/*.fastq.gz
ls results/qc_raw

## Step 5 of 16: Read the FastQC result
The HTML report is for looking at (download it, or see the figures at the end). The same results are also in text form in the folder FastQC made for each file (`liver_R1_fastqc/`), so we can read them in the shell.

**The traffic-light table:** every module gets PASS / WARN / FAIL for each of the four files (liver R1, liver R2, muscle R1, muscle R2). Do not panic about every FAIL: the thresholds are made for DNA-seq,
and RNA-seq is *expected* to fail some of them.

In [ ]:
%%bash
ls results/qc_raw/liver_R1_fastqc
cat results/qc_raw/{liver_R1,liver_R2,muscle_R1,muscle_R2}_fastqc/summary.txt | awk -F'\t' '{m[$2]=m[$2]"   "$1; if(!($2 in o)){o[$2]=++n; name[n]=$2}} END{printf "%-34s %s\n","module","liver_R1 liver_R2 muscle_R1 muscle_R2"; for(i=1;i<=n;i++) printf "%-34s %s\n", name[i], m[name[i]]}'

**How to read this table for RNA-seq**

| Module | What you see | Why |
|---|---|---|
| Per base sequence quality, per tile, per sequence quality | PASS | the reads are of high quality (Q37 for most bases) |
| **Per base sequence content** | FAIL | normal for RNA-seq: the first ~12 bases are biased because of the random primers used to make cDNA - it is **not** a sequencing problem |
| **Per sequence GC content** | WARN / FAIL | the reads come from a limited set of genes; when a few genes are very abundant, the GC curve is not a smooth bell. Worse in muscle |
| **Sequence duplication levels** | WARN / FAIL | a very abundant transcript is sampled again and again, so many reads are identical. In RNA-seq these are real, **do not remove duplicates** (muscle: about 69 % duplicated reads) |
| **Overrepresented sequences** | WARN | the same reason: pieces of the most abundant transcripts |
| Adapter content | PASS | the typical fragment (insert size peak about 160 bases, see fastp below) is longer than the read (101), so only a few reads reach the adapter |

The first lines of the data file `fastqc_data.txt` - the basic statistics of one file:

In [ ]:
%%bash
sed -n '1,12p' results/qc_raw/liver_R1_fastqc/fastqc_data.txt

The numbers behind **Per base sequence quality** (mean quality at each position of the read, first 12 positions):

In [ ]:
%%bash
sed -n '/>>Per base sequence quality/,/>>END_MODULE/p' results/qc_raw/liver_R1_fastqc/fastqc_data.txt | sed -n '1,14p'

## Step 6 of 16: MultiQC - all reports in one
With many samples you do not want to open one FastQC report per file. **MultiQC** reads all the reports in a folder and puts them into **one** HTML report (and one set of tables).
To look at the interactive HTML report, open the folder icon on the left of Colab, go to `RNAseq_calf/results/multiqc_raw`, and download `multiqc_raw.html` (right click > Download), then open it in your browser.

In [ ]:
%%bash
multiqc --no-ansi results/qc_raw -o results/multiqc_raw -n multiqc_raw --force
ls results/multiqc_raw results/multiqc_raw/multiqc_raw_data

One of the tables MultiQC made - the general statistics (percent duplicates, GC content, read length, number of reads in millions):

In [ ]:
%%bash
column -t -s $'\t' results/multiqc_raw/multiqc_raw_data/multiqc_general_stats.txt

## Step 7 of 16: fastp - trim the reads
**fastp** removes what should not be aligned: sequencing **adapters** (when a fragment is shorter than the read, the read runs into the adapter), **low-quality bases** at the end, and reads that become too short.
We know the kit (Illumina **TruSeq** stranded mRNA, see the sample description on GEO), so we give fastp its two adapter sequences. NovaSeq reads can also end in a run of `G` (no signal) - fastp removes that automatically.

| Option | Meaning |
|---|---|
| `-i` `-I` / `-o` `-O` | input / output of read 1 and read 2 |
| `--adapter_sequence` / `--adapter_sequence_r2` | the TruSeq adapter of read 1 / read 2 (both start with `AGATCGGAAGAGC`) |
| `--cut_tail --cut_tail_mean_quality 20` | cut the end of a read while the mean quality of a 4-base window is below 20 |
| `--length_required 36` | drop reads shorter than 36 bases after trimming |
| `--json` `--html` | reports (MultiQC reads the json) |

*Why not let fastp find the adapters itself?* It can (`--detect_adapter_for_pe`). We tried: on the liver sample it "detected" a 60-base sequence that is **not an adapter** but a piece of a very abundant liver transcript, cut it out of reads, and then discarded 1,338 reads as "too short" (with the right adapters: 22).
Automatic detection is handy when the kit is unknown, but always check what it found. With the right adapters this does not happen.

fastp prints a summary for every sample. Look at **reads passed filters**, **reads with adapter trimmed** and the **before / after** quality.

In [ ]:
%%bash
for s in liver muscle; do
  echo "=================== sample: $s"
  fastp -i data/reads/${s}_R1.fastq.gz -I data/reads/${s}_R2.fastq.gz \
        -o results/trim/${s}_R1.fastq.gz -O results/trim/${s}_R2.fastq.gz \
        --adapter_sequence AGATCGGAAGAGCACACGTCTGAACTCCAGTCA --adapter_sequence_r2 AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT \
        --cut_tail --cut_tail_mean_quality 20 --length_required 36 \
        --thread $THREADS --json results/trim/${s}.fastp.json --html results/trim/${s}.fastp.html --report_title $s
done

In [ ]:
%%bash
ls -lh results/trim

## Step 8 of 16: FastQC and MultiQC again - did trimming help?
Always check again after trimming. Same command, new input and a new output folder. Then one MultiQC report that holds **raw and trimmed** together.

In [ ]:
%%bash
fastqc -t $THREADS --extract --outdir results/qc_trim results/trim/*.fastq.gz
multiqc --no-ansi results/qc_raw results/qc_trim results/trim -o results/multiqc_trim -n multiqc_raw_and_trimmed --force

The traffic lights, **before and after trimming** (read 1 of each sample: raw, trimmed):

In [ ]:
%%bash
cat results/qc_raw/liver_R1_fastqc/summary.txt results/qc_trim/liver_R1_fastqc/summary.txt results/qc_raw/muscle_R1_fastqc/summary.txt results/qc_trim/muscle_R1_fastqc/summary.txt | awk -F'\t' '{m[$2]=m[$2]"   "$1; if(!($2 in o)){o[$2]=++n; name[n]=$2}} END{printf "%-34s %s\n","module","liver_raw liver_trim muscle_raw muscle_trim"; for(i=1;i<=n;i++) printf "%-34s %s\n", name[i], m[name[i]]}'

Not much changed - and that is a **good** result: these reads were already clean (high quality, few adapters), so fastp only removed a small number of reads and bases (compare the "before / after" numbers above; Figure 1 below shows the same).
The FAILs that remain (base content, GC, duplication) come from the biology of RNA-seq, not from the sequencer, so no trimming can remove them. Do not try to "fix" every FAIL: know why it fails.

## Step 9 of 16: HISAT2 - build the genome index
An aligner does not search the whole genome for every read. It first builds an **index** of the genome (like the index of a book) - once per genome, then used for every sample.
**HISAT2** is a *splice-aware* aligner: an RNA read can come from two exons, with an intron (thousands of bases) in between that is not in the read. HISAT2 can split such a read.
On chromosome 19 the index takes a minute or two (the whole genome: use a pre-built index from the HISAT2 website, or a bigger machine). The build log is long, so we show the start and the end.

In [ ]:
%%bash
hisat2-build -p $THREADS ref/chr19.fa index/chr19 > index/hisat2-build.log 2>&1
head -n 8 index/hisat2-build.log
tail -n 6 index/hisat2-build.log
ls -lh index

## Step 10 of 16: HISAT2 - align the reads
Each trimmed read pair is aligned to chromosome 19. HISAT2 writes SAM text; we send it straight into `samtools sort` (a pipe `|`), which sorts by position and writes a compact binary **BAM** file. Then `samtools index` makes the `.bai` index.
HISAT2 prints its alignment summary to the screen (stderr); we keep it in a log file, which MultiQC can read later.

In [ ]:
%%bash
for s in liver muscle; do
  echo "=================== sample: $s"
  hisat2 -p $THREADS -x index/chr19 -1 results/trim/${s}_R1.fastq.gz -2 results/trim/${s}_R2.fastq.gz 2> results/align/${s}.hisat2.log | samtools sort -@ 2 -o results/align/${s}.bam -
  samtools index results/align/${s}.bam
done
ls -lh results/align

The HISAT2 summaries. **Overall alignment rate** is the percentage of reads that found a place in the genome.
*Note:* these reads were chosen because they come from chromosome 19, so the rate is much higher than in a real experiment (in a real run 85-95 % of the reads align).

In [ ]:
%%bash
cat results/align/liver.hisat2.log
cat results/align/muscle.hisat2.log

## Step 11 of 16: Look inside the BAM file
A BAM file is binary; `samtools view` shows it as text. First the **header** (the chromosome name and length, and the commands that made the file), then three alignments.
Columns: 1 read name, 2 FLAG, 3 chromosome, 4 position, 5 mapping quality, 6 CIGAR, 7-9 mate chromosome / position / insert size; then optional tags (NH = number of places the read aligns, ...).

In [ ]:
%%bash
samtools view -H results/align/liver.bam
samtools view results/align/liver.bam | head -n 3 | cut -f1-9,12-

The **FLAG** is a number that packs yes/no answers. What do 99 and 147 mean (a normal pair: one read on the forward strand, its mate on the reverse strand)?

In [ ]:
%%bash
samtools flags 99 147

**CIGAR** describes how the read matches the genome: `101M` = 101 bases match; `40M2000N61M` = 40 bases match, then the genome continues **2,000 bases (N = intron)** without the read, then 61 bases match.
Reads that cross an intron are the signature of RNA-seq. How many alignments are there, how many have an `N` in their CIGAR string, and what do four typical ones look like
(unique alignments, MAPQ 60, an intron of 100-9,999 bases; columns: read, FLAG, chromosome, start, mapping quality, CIGAR, mate chromosome, mate start, insert size)?
Note that the insert size of an RNA pair is measured on the genome, so an intron between the two reads makes it larger than the RNA fragment really was.

In [ ]:
%%bash
samtools view -c results/align/liver.bam
samtools view results/align/liver.bam | awk '$6 ~ /N/' | wc -l
samtools view results/align/liver.bam | awk '$5==60 && $9>0 && $9<5000 && $6 ~ /^[0-9]+M[0-9][0-9][0-9][0-9]?N[0-9]+M$/' | head -n 4 | cut -f1-9

## Step 12 of 16: Alignment QC
`samtools flagstat` counts the reads by category (mapped, paired, properly paired, duplicates, ...). `samtools stats` gives many more numbers; we keep them in files and show the summary lines.

In [ ]:
%%bash
for s in liver muscle; do
  samtools flagstat results/align/$s.bam > results/align/$s.flagstat.txt
  samtools stats results/align/$s.bam > results/align/$s.stats.txt
done
cat results/align/liver.flagstat.txt
cat results/align/muscle.flagstat.txt

In [ ]:
%%bash
grep ^SN results/align/liver.stats.txt | cut -f2,3 | sed -n '1,8p;12,13p;15,17p;20,22p'

## Step 13 of 16: Which strand? (library strandedness)
Many RNA-seq kits are **stranded**: the read tells you from which DNA strand the RNA was copied. featureCounts must be told which kind of library you have (`-s 0` = not stranded,
`-s 1` = read 1 on the gene's strand, `-s 2` = read 1 on the opposite strand, "reverse stranded", e.g. Illumina TruSeq stranded and dUTP kits). Wrong choice = most reads are not counted or counted for the wrong gene.

If the kit is not known, **try all three** on one sample (we use liver only, to save time) and compare the number of **Assigned** read pairs:

| Result | Library is |
|---|---|
| `-s 1` and `-s 2` each get about half of `-s 0` | **not stranded** |
| `-s 2` is about as high as `-s 0`, `-s 1` is tiny | **reverse stranded** (use `-s 2`) |
| `-s 1` is about as high as `-s 0`, `-s 2` is tiny | **forward stranded** (use `-s 1`) |

(`-s 0` counts a read pair for a gene on either strand, so it is always high; it also gives more "Ambiguity", because two genes on opposite strands overlap.)

In [ ]:
%%bash
for s in 0 1 2; do
  featureCounts -T $THREADS -p --countReadPairs -s $s -t exon -g gene_id -a data/ref/chr19.gtf.gz -o results/counts/test_s$s.txt results/align/liver.bam > /dev/null 2>&1
  echo "-s $s"; grep -E '^(Assigned|Unassigned_NoFeatures|Unassigned_Ambiguity)' results/counts/test_s$s.txt.summary
done

## Step 14 of 16: featureCounts - count read pairs per gene
**featureCounts** takes the BAM files and the annotation and counts, for every gene, the read pairs that overlap its exons.

| Option | Meaning |
|---|---|
| `-T` | threads |
| `-p --countReadPairs` | the data are paired-end: count read **pairs** (fragments), not single reads |
| `-s 2` | reverse-stranded library (the result of step 13) |
| `-t exon` | count reads that overlap `exon` lines of the GTF |
| `-g gene_id` | and add them up per `gene_id` |
| `--extraAttributes gene_name` | also write the gene name into the table |
| `-a` / `-o` | annotation (GTF) / output file |

All BAM files go into **one** call, so that the output is already one table with one column per sample.

In [ ]:
%%bash
featureCounts -T $THREADS -p --countReadPairs -s 2 -t exon -g gene_id --extraAttributes gene_name -a data/ref/chr19.gtf.gz -o results/counts/counts.txt results/align/liver.bam results/align/muscle.bam

The summary says what happened to every read pair: **Assigned** (counted for a gene) or not (no feature = the pair lies between genes or in an intron, ambiguity = overlaps two genes, ...).

In [ ]:
%%bash
cat results/counts/counts.txt.summary

**One last MultiQC report for the whole pipeline**: FastQC (raw and trimmed), fastp, HISAT2, samtools and featureCounts, all in one HTML file you can hand to a colleague.
(MultiQC lists the HISAT2 logs as "bowtie2": the two aligners write the same kind of summary.)

In [ ]:
%%bash
multiqc --no-ansi results/qc_raw results/qc_trim results/trim results/align results/counts/counts.txt.summary -o results/multiqc_final -n multiqc_final --force

## Step 15 of 16: Make the gene count matrix
The featureCounts file has a comment line first, then a header and one row per gene: gene_id, chromosome, start, end, strand, length, gene_name, and one column per BAM file.
We turn it into the **gene count matrix** that every downstream tool (DESeq2, edgeR, limma ...) wants: **rows = genes, columns = samples, cells = number of read pairs (raw counts, integers)**.
First the raw file:

In [ ]:
%%bash
head -n 5 results/counts/counts.txt | cut -f1,2,6-

Now the clean matrix. `grep -v '^#'` drops the comment line, `cut` keeps the gene id and the sample columns, `sed` shortens the column names from the file path to the sample name.
A second, annotated version has the gene name and the gene length too (many cattle genes have no symbol - featureCounts writes `NA` - and get their Ensembl ID instead).

In [ ]:
%%bash
grep -v '^#' results/counts/counts.txt | cut -f1,8- | sed '1s#results/align/##g; 1s#\.bam##g' > results/gene_count_matrix.tsv
grep -v '^#' results/counts/counts.txt | awk -F'\t' -v OFS='\t' 'NR==1{print "gene_id","gene_name","length","liver","muscle"; next} {print $1, ($7=="NA"?$1:$7), $6, $8, $9}' > results/gene_count_matrix_annotated.tsv
head -n 6 results/gene_count_matrix.tsv
head -n 6 results/gene_count_matrix_annotated.tsv

## Step 16 of 16: Look at the matrix
First sanity checks: how many genes, how many read pairs were counted per sample (the **library size**), how many genes have any counts.

In [ ]:
%%bash
echo "genes (rows): $(( $(wc -l < results/gene_count_matrix.tsv) - 1 ))"
awk -F'\t' 'NR>1 {l+=$2; m+=$3; if($2>0) gl++; if($3>0) gm++; if($2>0 && $3>0) both++; if($2+$3>=10) ten++} END{print "counted read pairs  liver:", l, " muscle:", m; print "genes with counts   liver:", gl, " muscle:", gm, " both:", both; print "genes with >= 10 counts in total:", ten}' results/gene_count_matrix.tsv

The 8 most-counted genes in each tissue (columns: gene id, gene name, gene length, liver, muscle):

In [ ]:
%%bash
echo "--- liver"
sort -t$'\t' -k4,4nr results/gene_count_matrix_annotated.tsv | head -n 8 | column -t -s $'\t'
echo "--- muscle"
sort -t$'\t' -k5,5nr results/gene_count_matrix_annotated.tsv | head -n 8 | column -t -s $'\t'

Liver or muscle? Look up some known **liver** genes (VTN vitronectin, APOH apolipoprotein H, ASGR1 asialoglycoprotein receptor, G6PC1 glucose-6-phosphatase) and **muscle** genes (MYH1 and MYH2 myosin heavy chains, ENO3 enolase 3, SCN4A, CACNG1 - muscle ion channels):

In [ ]:
%%bash
head -n 1 results/gene_count_matrix_annotated.tsv
grep -w -E 'VTN|APOH|ASGR1|G6PC1|MYH1|MYH2|ENO3|SCN4A|CACNG1' results/gene_count_matrix_annotated.tsv | sort -t$'\t' -k2,2 | column -t -s $'\t'

**Raw counts cannot be compared between samples directly:** muscle has more counted read pairs than liver, and a long gene gets more reads than a short one. A first simple fix is
**CPM** = counts per million counted read pairs (it corrects for library size, not for gene length). DESeq2 and edgeR use better methods; this is only to look.

In [ ]:
%%bash
awk -F'\t' -v OFS='\t' 'NR==FNR{ if(FNR>1){l+=$4; m+=$5} next } FNR==1{print; next} {printf "%s\t%s\t%s\t%.1f\t%.1f\n", $1,$2,$3,$4*1e6/l,$5*1e6/m}' results/gene_count_matrix_annotated.tsv results/gene_count_matrix_annotated.tsv > results/gene_cpm_annotated.tsv
head -n 5 results/gene_cpm_annotated.tsv | column -t -s $'\t'

# PART C - Figures
These cells are Python (pandas + matplotlib). They only **draw** the tables and logs that the shell commands made above.

**Figure 1 - read quality before and after fastp.** Mean quality at every position of read 1 and read 2 (from the fastp json files). The light (before) and dark (after) lines lie almost on top of each other: the reads were already clean, so trimming changed little. Note the y axis: a mean quality of 36 is very high.

In [ ]:
import json
import matplotlib.pyplot as plt

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8d8d89"      # liver, muscle, neutral
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "axes.facecolor": "white", "figure.facecolor": "white"})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, s, col in zip(axes, ["liver", "muscle"], [BLUE, ORANGE]):
    j = json.load(open(f"results/trim/{s}.fastp.json"))
    for r, ls in (("read1", "-"), ("read2", ":")):
        for when, alpha in (("before", 0.35), ("after", 1.0)):
            q = j[f"{r}_{when}_filtering"]["quality_curves"]["mean"]
            ax.plot(range(1, len(q) + 1), q, ls, color=col, alpha=alpha, lw=1.8)
    ax.axhline(30, color=GREY, lw=0.8, ls="--")
    ax.text(1, 30.25, "Q30", fontsize=8.5, color="#52514e")
    ax.set_ylim(28, 38)
    ax.set_title(f"{s}", loc="left", fontweight="bold", color=col)
    ax.set_xlabel("position in read (bases)")
    ax.grid(axis="y", color="#e6e6e3", lw=0.8)
    ax.text(0.98, 0.06, "solid = read 1, dotted = read 2\nlight = before fastp, dark = after", transform=ax.transAxes, ha="right", fontsize=8.5, color="#52514e")
axes[0].set_ylabel("mean base quality (Phred)")
fig.suptitle("Figure 1. Base quality along the reads, before and after fastp", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

**Figure 2 - what happened to the reads?** Left: HISAT2 (how many read pairs aligned). Right: featureCounts (how many aligned pairs could be given to a gene).

In [ ]:
import re

def hisat2(path):
    t = open(path).read()
    n = lambda pat: int(re.search(pat, t).group(1))
    total = n(r"(\d+) reads; of these")
    zero  = n(r"(\d+) \([\d.]+%\) aligned concordantly 0 times")
    one   = n(r"(\d+) \([\d.]+%\) aligned concordantly exactly 1 time")
    many  = n(r"(\d+) \([\d.]+%\) aligned concordantly >1 times")
    return total, [("aligned once", one), ("aligned >1 places", many), ("not aligned as a pair", zero)]

def fcounts(path):
    rows = [l.rstrip("\n").split("\t") for l in open(path)][1:]
    return {r[0]: [int(x) for x in r[1:]] for r in rows}

fc = fcounts("results/counts/counts.txt.summary")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
cols = [BLUE, "#7fb0ec", GREY]
for k, s in enumerate(["liver", "muscle"]):
    total, parts = hisat2(f"results/align/{s}.hisat2.log")
    left = 0
    for (lab, v), c in zip(parts, cols):
        axes[0].barh(1 - k, v / total * 100, left=left, color=c, edgecolor="white", height=0.55, label=lab if k == 0 else None)
        if v / total > 0.12: axes[0].text(left + v / total * 50, 1 - k, f"{v / total * 100:.1f}%", ha="center", va="center", color="white" if c != "#7fb0ec" else "#0b0b0b", fontsize=9)
        left += v / total * 100
    tot = sum(fc[x][k] for x in fc)
    left = 0
    for lab, c, name in (("Assigned", BLUE, "counted for a gene"), ("Unassigned_MultiMapping", "#8d8d89", "not counted: aligns in more than one place"),
                         ("Unassigned_NoFeatures", "#c9c9c5", "not counted: no gene (intron, between genes)"), ("Unassigned_Ambiguity", "#e5e5e2", "not counted: overlaps two genes")):
        v = fc.get(lab, [0, 0])[k]
        axes[1].barh(1 - k, v / tot * 100, left=left, color=c, edgecolor="white", height=0.55, label=name if k == 0 else None)
        if v / tot > 0.12: axes[1].text(left + v / tot * 50, 1 - k, f"{v / tot * 100:.1f}%", ha="center", va="center", color="white" if c in (BLUE, "#8d8d89") else "#0b0b0b", fontsize=9)
        left += v / tot * 100
for ax, t, nc in zip(axes, ["HISAT2: read pairs aligned to chr19", "featureCounts: what happened to the aligned pairs"], [2, 2]):
    ax.set_yticks([1, 0]); ax.set_yticklabels(["liver", "muscle"]); ax.set_xlim(0, 100); ax.set_xlabel("% of read pairs" if ax is axes[0] else "% of alignments")
    ax.set_title(t, loc="left", fontsize=10); ax.legend(loc="upper left", bbox_to_anchor=(0, -0.3), ncol=nc, frameon=False, fontsize=8.5)
fig.suptitle("Figure 2. Alignment and counting", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

**Figure 3 - liver against muscle, gene by gene.** Every dot is one gene; position = counts per million (log scale). Genes on the diagonal are expressed equally; genes far from it are tissue-specific.

In [ ]:
import numpy as np, pandas as pd

cpm = pd.read_csv("results/gene_cpm_annotated.tsv", sep="\t", keep_default_na=False)
cnt = pd.read_csv("results/gene_count_matrix_annotated.tsv", sep="\t", keep_default_na=False)     # same genes, same order, raw counts
cpm["lx"], cpm["ly"] = np.log10(cpm["liver"] + 1), np.log10(cpm["muscle"] + 1)
cpm["d"] = cpm["ly"] - cpm["lx"]
fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.scatter(cpm.lx, cpm.ly, s=9, color="#c9c9c5", lw=0, label="all genes")
liv, mus = cpm[(cpm.d < -1) & (cnt.liver >= 20)], cpm[(cpm.d > 1) & (cnt.muscle >= 20)]      # > 10x higher and at least 20 read pairs
ax.scatter(liv.lx, liv.ly, s=14, color=BLUE, lw=0, label=f"liver > 10x muscle  ({len(liv)} genes)")
ax.scatter(mus.lx, mus.ly, s=14, color=ORANGE, lw=0, label=f"muscle > 10x liver  ({len(mus)} genes)")
lim = max(cpm.lx.max(), cpm.ly.max()) + 0.15
ax.plot([0, lim], [0, lim], color=GREY, lw=0.8, ls="--")
for name, off in {"VTN": (6, 4), "APOH": (6, -3), "ABCA10": (-44, -14), "MYH1": (7, -3), "MYH2": (7, -3), "ENO3": (8, -3), "SMTNL2": (6, 4)}.items():
    r = cpm[cpm.gene_name == name].iloc[0]
    ax.annotate(name, (r.lx, r.ly), xytext=off, textcoords="offset points", fontsize=8.5)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("liver, log10(CPM + 1)"); ax.set_ylabel("muscle, log10(CPM + 1)")
ax.legend(loc="upper left", frameon=False, fontsize=8.5, bbox_to_anchor=(0.16, 1.0))
ax.set_title("Figure 3. Gene expression of calf liver and muscle (chromosome 19)", loc="left", fontsize=10.5)
plt.tight_layout()
plt.show()

**Figure 4 - the most-counted genes of each tissue** (raw counts of read pairs). A few genes take a big share of all reads - in muscle one myosin gene alone.

In [ ]:
ann = pd.read_csv("results/gene_count_matrix_annotated.tsv", sep="\t", keep_default_na=False)
ann["label"] = ann.gene_name.where(~ann.gene_name.str.startswith("ENSBTAG"), "ENSBTAG..." + ann.gene_name.str[-5:])     # genes without a symbol: shortened Ensembl ID
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, s, col in zip(axes, ["liver", "muscle"], [BLUE, ORANGE]):
    top = ann.nlargest(10, s).iloc[::-1]
    ax.barh(top.label, top[s], color=col, height=0.6)
    for y, v in enumerate(top[s]): ax.text(v, y, f" {v:,}", va="center", fontsize=8.5, color="#52514e")
    share = top[s].sum() / ann[s].sum() * 100
    ax.set_title(f"{s}: top 10 genes = {share:.0f}% of counted pairs", loc="left", fontsize=10, color=col, fontweight="bold")
    ax.set_xlim(0, top[s].max() * 1.22); ax.set_xlabel("read pairs counted")
fig.suptitle("Figure 4. Most-counted genes", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

**Figure 5 - where do the reads fall on the chromosome?** Read pairs counted per 1 Mb of chromosome 19 (gene start used as the position).

In [ ]:
raw = pd.read_csv("results/counts/counts.txt", sep="\t", comment="#")
raw["pos"] = raw["Start"].astype(str).str.split(";").str[0].astype(int)
bins = np.arange(0, raw.pos.max() + 1_000_000, 1_000_000)
fig, ax = plt.subplots(figsize=(11, 3.2))
for col, c, off in (("results/align/liver.bam", BLUE, -0.2), ("results/align/muscle.bam", ORANGE, 0.2)):
    h, _ = np.histogram(raw.pos, bins=bins, weights=raw[col])
    ax.bar(bins[:-1] / 1e6 + 0.5 + off, h, width=0.4, color=c, label=col.split("/")[-1][:-4])
ax.annotate("MYH1 + MYH2\n(myosin genes)", (bins[h.argmax()] / 1e6 + 0.7, h.max()), xytext=(8, -4), textcoords="offset points", fontsize=8.5, color=ORANGE, va="top")   # h = muscle, the last bars drawn
ax.set_yscale("log"); ax.set_xlabel("position on chromosome 19 (Mb)"); ax.set_ylabel("read pairs per Mb (log)")
ax.legend(frameon=False, ncol=2, loc="upper right")
ax.set_title("Figure 5. Counted read pairs along chromosome 19", loc="left", fontsize=10.5)
plt.tight_layout()
plt.show()

# PART D - Wrap-up

## What did we make?

In [ ]:
%%bash
ls -lh results
du -sh results/*  | sort -h

`results/gene_count_matrix.tsv` is the file you carry to the next step. Optional: download all results as one zip (reports, BAM files, count tables).

In [ ]:
import shutil
zipfile = shutil.make_archive(os.path.join(os.path.dirname(WORK), "RNAseq_calf_results"), "zip", WORK, "results")
print(zipfile, f"{os.path.getsize(zipfile)/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zipfile)
except ImportError:
    print("(not in Colab: the zip is in the folder above)")

## Questions
1. How many read pairs are in each FASTQ file, and how many of them survived fastp? Which kind of read was removed?
2. Which FastQC modules fail for these RNA-seq reads? (Hint: *Per base sequence content*, *Sequence duplication levels* - why are these *expected* for RNA-seq?)
3. HISAT2 reports an alignment rate above 99 % here. Why is this too good to be true for a real experiment?
4. In step 13, what did the three `-s` settings give, and what does it tell you about the library? What would have happened with the wrong stranded setting?
5. Liver and muscle have a different number of counted read pairs. Why can you not compare the raw counts of one gene between the two columns?
6. In muscle, a single gene has more than half of all counts. What does this do to the other genes when we calculate CPM? (This is why DESeq2 and edgeR use a better normalisation than CPM.)

## Where to go next
* **Replicates are needed for statistics.** This matrix has one sample per tissue - enough to see the biology, not to test it. With 3+ samples per group, `results/gene_count_matrix.tsv` goes straight into **DESeq2** or **edgeR** (differential expression), then volcano plots, PCA and GO/pathway enrichment.
* **Whole genome:** replace chromosome 19 by the whole ARS-UCD2.0 genome (use a pre-built HISAT2 or STAR index, 8-32 GB RAM) and run the same commands for every sample in a loop (`for s in ...`).
* **Other ways to count:** alignment-free quantification (Salmon, kallisto) skips steps 9-14 and counts straight from the reads.

## Data sources and tools
* Reads: GEO **GSE289870** / ENA **PRJNA1225222** (University College Dublin), runs SRR32383427 (liver, GSM8800637) and SRR32383411 (muscle, GSM8800653); the first 100,000 read pairs that align to chromosome 19, unchanged. See `data/SOURCES.md` and `data/make_subset.sh`.
* Reference and annotation: Ensembl release 114, *Bos taurus* ARS-UCD2.0, chromosome 19.
* Tools: FastQC 0.13.0, MultiQC 1.35, fastp 1.4.0, HISAT2 2.2.3, samtools 1.24, Subread (featureCounts) 2.1.1.